
# Data Skew in Apache Spark: A Practical Guide

## What is Data Skew?

**Data skew** is when data is unevenly distributed across Spark partitions.

Ideally, 1 billion rows split into 100 partitions means **10 million rows per partition**. Skew happens when some partitions get much more data:


No Skew:
Partition 1: ██████████ 10M rows
Partition 2: ██████████ 10M rows
Partition 3: ██████████ 10M rows

Skewed:
Partition 1: ██ 2M rows
Partition 2: ██ 2M rows
Partition 3: █████████████████████████████ 92M rows ← Skew!


---

## Why Does Skew Occur?

### Example

Suppose you group an orders table by `country`:


USA     → 500M rows   ← dominant
Germany → 10M rows
France  → 8M rows
Brazil  → 7M rows
India   → 6M rows


All USA rows go to one partition, causing that executor to process 500M rows while others handle much less.

### Common Causes

- **Dominant key values** (e.g., USA, NULL)
- **NULLs** all hash to one partition
- **Joins** with popular keys (e.g., product_id="iPhone")
- **Filtering** leaves one partition with most data
- **Poor partition column** (e.g., gender: only M/F)

---

## Detecting Skew

### Using Spark UI

Check **Stages → Task Metrics**:


Task ID | Duration | Shuffle Read | Records
Task 1  | 2s  | 50MB  | 500K
Task 2  | 2s  | 48MB  | 490K
Task 3  | 180s| 4.5GB | 45M   ← Skew!


**Warning signs:**
- One task much slower/larger than others
- Huge difference between median and max task duration

### Key Distribution

Check if one value dominates:


country   | row_count
USA       | 45M   ← dominant
Germany   | 500K
France    | 480K


### Partition Sizes


partition_id | row_count
5            | 45M   ← Skewed
1            | 500K


---

## Analogy

Spark tasks are like **cashiers**:


Cashier 1: 2 customers (done in 2 min)
Cashier 10: 500 customers (takes 2 hours)


Store closes only when the busiest cashier finishes. In Spark, skew means most executors sit idle while one works overtime.

---

## How to Fix Skew

### 1. Enable AQE (Adaptive Query Execution)

Spark 3+ can auto-detect and split skewed partitions:


spark.sql.adaptive.enabled = true
spark.sql.adaptive.skewJoin.enabled = true
spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 256MB


*Works for joins/aggregations.*

---

### 2. Broadcast Join

If one table is small, broadcast it:


spark.sql.autoBroadcastJoinThreshold = 100MB


Or use SQL hint:

sql
SELECT /*+ BROADCAST(products) */ *
FROM orders
JOIN products ON orders.product_id = products.product_id


---

### 3. Salting

Add a random salt to the skewed key to split it:


Before: country=USA → 1 partition → 500M rows
After:  country=USA_0, USA_1, ..., USA_9 → 10 partitions → 50M rows each


Replicate the lookup table for each salt value.

---

### 4. Handle NULLs Separately

Split data:


df_valid = WHERE customer_id IS NOT NULL
df_nulls = WHERE customer_id IS NULL

result_valid = df_valid JOIN customers
result_nulls = df_nulls WITH customer_name="UNKNOWN"

final = result_valid UNION result_nulls


---

### 5. Two-Phase Aggregation

Aggregate locally first, then globally:


Phase 1: Partial sums per salt (USA_0, USA_1, ...)
Phase 2: Combine partial sums for USA


---

### 6. Repartition on a Better Column

Choose a column with uniform distribution or use round-robin:


REPARTITION BY order_id
REPARTITION 200


---

## Fix Comparison

| Fix                | Best For                | Effort | Limitation                |
|--------------------|------------------------|--------|---------------------------|
| AQE                | Joins & aggregations   | Zero   | Spark 3+, not all cases   |
| Broadcast join     | Small table joins      | Low    | Table must fit in memory  |
| NULL handling      | NULL skew              | Low    | Only fixes NULL skew      |
| Two-phase agg      | GroupBy skew           | Medium | Only aggregations         |
| Salting            | Large skewed joins     | Medium | Increases table size      |
| Repartition        | Post-filter skew       | Low    | May cause extra shuffle   |

---

## Key Takeaways

- **Skew = Uneven data across partitions**
- **Symptoms:** Slow tasks, job hangs, high resource usage
- **Causes:** Dominant keys, NULLs, poor partitioning
- **Fixes:** AQE, broadcast join, NULL handling, two-phase aggregation, salting, repartition

**Bottom line:** Skew kills parallelism. Fix it to unlock Spark's speed!

In [0]:
# What is Skew?
# Skew is the unequal distribution of data across partitions in Spark.
# If one partition has much more data than others, it slows down the job because all tasks must finish.

# Why does skew happen?
# - Dominant key values (e.g., USA, NULL, popular product IDs)
# - Grouping or joining on non-uniform columns
# - Poor partition column choice

# How to detect skew:
from pyspark.sql.functions import spark_partition_id, count, stddev, mean

# Check partition sizes
display(
    df.groupBy(spark_partition_id().alias("pid"))
      .count()
      .orderBy("count", ascending=False)
)

# Check distribution of join/group key
display(
    df.groupBy("country")
      .count()
      .orderBy("count", ascending=False)
)

# Check standard deviation of partition sizes
partition_counts = df.groupBy(spark_partition_id().alias("pid")).agg(count("*").alias("cnt"))
display(
    partition_counts.select(
        mean("cnt").alias("avg_rows"),
        stddev("cnt").alias("stddev_rows")
    )
)

# How to handle skew:

# 1. Enable AQE (Adaptive Query Execution)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB")

# 2. Broadcast join if one side is small
from pyspark.sql.functions import broadcast
result = large_orders.join(broadcast(small_products), "product_id")

# 3. Handle NULL keys separately
from pyspark.sql.functions import col, lit
df_null_keys = df.filter(col("customer_id").isNull())
df_valid_keys = df.filter(col("customer_id").isNotNull())
result_valid = df_valid_keys.join(customers, "customer_id")
result_nulls = df_null_keys.withColumn("customer_name", lit("UNKNOWN"))
final_result = result_valid.union(result_nulls)

# 4. Salting for skewed joins
from pyspark.sql.functions import rand, concat, floor, explode, array
SALT_FACTOR = 10
df_large_salted = df_large.withColumn("salt", (floor(rand() * SALT_FACTOR)).cast("int")) \
                          .withColumn("salted_key", concat(col("country"), lit("_"), col("salt")))
df_small_replicated = df_small.withColumn("salt_array", array([lit(i) for i in range(SALT_FACTOR)])) \
                              .withColumn("salt", explode(col("salt_array"))) \
                              .withColumn("salted_key", concat(col("country"), lit("_"), col("salt"))) \
                              .drop("salt_array", "salt")
result = df_large_salted.join(df_small_replicated, "salted_key").drop("salted_key")

# 5. Two-phase aggregation
from pyspark.sql.functions import sum
SALT = 10
phase1 = df.withColumn("salt", (floor(rand() * SALT)).cast("int")) \
           .groupBy("country", "salt") \
           .agg(sum("sales").alias("partial_sum"))
phase2 = phase1.groupBy("country").agg(sum("partial_sum").alias("total_sales"))

# 6. Repartition on a better column
df_repartitioned = df.repartition(200, "order_id")  # Use a uniformly distributed column